# 02. Pollution & DSC (Text Cell)

ADR-016 §3-3 / ADR-017 §3-3 polluter 7종 × 6 level × 6 dataset 스윕 → DSC 점수 + polluter quality measure 수집.

Output: `results/text_cell_dsc_sweep.csv` (rows = dataset × polluter × level × seed)

---


## 0. import + 데이터 (01에서 캐싱)

새 세션이면 drive 재마운트 필요. 같은 런타임에서 01 다음으로 실행이면 이 셀 skip 가능.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import sys, os, glob, json
import numpy as np
import pandas as pd


def _find_dsc_base():
    root = '/content/drive/MyDrive'
    if not os.path.isdir(root):
        return None
    for c in [f'{root}/capstone/dsc', f'{root}/dsc', f'{root}/capstone-dsc']:
        if os.path.isfile(f'{c}/dsc_framework/__init__.py'):
            return c
    for pat in [f'{root}/*/dsc_framework/__init__.py',
                f'{root}/*/*/dsc_framework/__init__.py',
                f'{root}/*/*/*/dsc_framework/__init__.py']:
        for hit in glob.glob(pat):
            return os.path.dirname(os.path.dirname(hit))
    return None


BASE = _find_dsc_base()
if BASE is None:
    raise RuntimeError(
        'dsc_framework 폴더 못 찾음. 01 노트북 0-1 셀의 진단 메시지 + '
        'G드라이브 sync 상태 확인.'
    )
if BASE not in sys.path:
    sys.path.insert(0, BASE)
RESULTS_DIR = f'{BASE}/results'

from dsc_framework.text_cell import compute_dsc_text
from dsc_framework.text_cell_regression import compute_dsc_text_regression
from dsc_framework.text_polluters import (
    CompletenessTextPolluter, NoiseInjectionTextPolluter, WordShufflePolluter,
    ClassBalanceTextPolluter, LabelSwapTextPolluter,
    TargetDistributionSkewTextPolluter, TargetNoiseTextPolluter,
)
print(f'BASE: {BASE}')


## 1. 스윕 설정 (ADR-016/017 freeze)


In [ ]:
LEVEL_GRID = [0.0, 0.1, 0.25, 0.5, 0.75, 0.9]  # 6 단계, ADR-016 §3-3
SEEDS = [42, 7, 99]                                  # 3 seed, polluter hold-out 분석에 사용

CLASSIFICATION_POLLUTERS = {
    'completeness_text':    CompletenessTextPolluter,
    'noise_injection_text': NoiseInjectionTextPolluter,
    'word_shuffle':         WordShufflePolluter,
    'class_balance':        ClassBalanceTextPolluter,
    'label_swap':           LabelSwapTextPolluter,
}
REGRESSION_POLLUTERS = {
    'completeness_text':         CompletenessTextPolluter,
    'noise_injection_text':      NoiseInjectionTextPolluter,
    'word_shuffle':              WordShufflePolluter,
    'target_distribution_skew':  TargetDistributionSkewTextPolluter,
    'target_noise':              TargetNoiseTextPolluter,
}


## 2. 분류 트랙 스윕


In [ ]:
# datasets은 01 노트북에서 로드된 변수 그대로 사용 (Colab 메모리 공유)
def sweep_classification(texts, labels, dataset_name, polluters, seeds=SEEDS, levels=LEVEL_GRID):
    rows = []
    for pol_name, pol_cls in polluters.items():
        for seed in seeds:
            for lvl in levels:
                pol = pol_cls(lvl, random_seed=seed)
                t_p, l_p = pol.pollute(texts, labels)
                r = compute_dsc_text(t_p, l_p, use_embeddings=True,
                                     sample_cap=1000, random_state=seed)
                rows.append({
                    'dataset': dataset_name, 'polluter': pol_name,
                    'level': lvl, 'seed': seed,
                    'dsc_score': r['score'], **{k: r[k] for k in r if k not in ('score', 'grade')}
                })
    return pd.DataFrame(rows)


In [ ]:
# 실제 sweep — Colab GPU에서 실행. dataset 당 ~10분 (DistilBERT embedding 추출 포함)
# results_cls = []
# for name, ds in [('ag_news', ag_news['train']), ('imdb', imdb['train']), ('20news', news20['train'])]:
#     texts, labels = ds['text'][:5000], ds['label'][:5000]
#     df = sweep_classification(texts, labels, name, CLASSIFICATION_POLLUTERS)
#     results_cls.append(df)
# df_cls = pd.concat(results_cls); df_cls.to_csv('results/text_cls_dsc_sweep.csv', index=False)


## 3. 회귀 트랙 스윕


In [ ]:
def sweep_regression(texts, targets, dataset_name, polluters, seeds=SEEDS, levels=LEVEL_GRID):
    rows = []
    for pol_name, pol_cls in polluters.items():
        for seed in seeds:
            for lvl in levels:
                pol = pol_cls(lvl, random_seed=seed)
                t_p, tg_p = pol.pollute(texts, targets)
                r = compute_dsc_text_regression(t_p, tg_p, use_embeddings=True,
                                                sample_cap=1000, random_state=seed)
                rows.append({
                    'dataset': dataset_name, 'polluter': pol_name,
                    'level': lvl, 'seed': seed,
                    'dsc_score': r['score'], **{k: r[k] for k in r if k not in ('score', 'grade')}
                })
    return pd.DataFrame(rows)


In [ ]:
# 실제 sweep
# results_reg = []
# for name, ds in [('yelp_50k', yelp_train_s), ('amazon_200k', amazon_train_s),
#                  ('sst5', pd.DataFrame(sst5['train']))]:
#     texts = ds['text'].tolist(); targets = [float(x) for x in ds['label'].tolist()]
#     df = sweep_regression(texts[:5000], targets[:5000], name, REGRESSION_POLLUTERS)
#     results_reg.append(df)
# df_reg = pd.concat(results_reg); df_reg.to_csv('results/text_reg_dsc_sweep.csv', index=False)


---

다음: `03_training_text.ipynb` — 모델 학습 + accuracy/R² 수집.
